In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# مراحل المُحوِّل (Transpiler)

<details>
<summary><b>إصدارات الحزم</b></summary>

الكود في هذه الصفحة طُوِّر باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
تصف هذه الصفحة مراحل خط أنابيب التحويل المُعدّ مسبقًا في Qiskit SDK. هناك ست مراحل:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

تُنشئ دالة [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) [مدير مراحل مُعدّ مسبقًا](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) مؤلَّفًا من هذه المراحل. تعتمد التمريرات المحددة التي تشكّل كل مرحلة على الوسائط الممررة إلى `generate_preset_pass_manager`. الوسيطة `optimization_level` هي وسيطة موضعية يجب تحديدها؛ وهي عدد صحيح يمكن أن يكون 0 أو 1 أو 2 أو 3. تشير القيم الأعلى إلى تحسين أقوى لكنه أكثر تكلفة (انظر [الإعدادات الافتراضية لخيارات التحويل والضبط](defaults-and-configuration-options)).

الطريقة الموصى بها لتحويل دائرة هي إنشاء مدير مراحل مُعدّ مسبقًا ثم تشغيله على الدائرة، كما هو موضح في [التحويل باستخدام مديري التمريرات](transpile-with-pass-managers). ومع ذلك، يوجد بديل أبسط وإن كان أقل قابلية للتخصيص، وهو استخدام دالة [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). تقبل هذه الدالة الدائرة مباشرةً كوسيطة. كما هو الحال مع `generate_preset_pass_manager`، تعتمد تمريرات المُحوِّل المستخدمة على الوسائط، مثل `optimization_level`، الممررة إلى `transpile`. في الواقع، تستدعي دالة `transpile` داخليًا `generate_preset_pass_manager` لإنشاء مدير مراحل مُعدّ مسبقًا وتشغيله على الدائرة.
## مرحلة Init
لا تفعل هذه المرحلة الأولى الكثير بشكل افتراضي، وتكون مفيدة بشكل أساسي إذا أردت تضمين تحسيناتك الأولية الخاصة. نظرًا لأن معظم خوارزميات التخطيط والتوجيه مُصمَّمة للعمل مع بوابات الكيوبت الواحد والكيوبتين فقط، تُستخدم هذه المرحلة أيضًا لترجمة أي بوابات تعمل على أكثر من كيوبتين إلى بوابات تعمل على كيوبت أو كيوبتين فقط.

لمزيد من المعلومات حول تطبيق تحسيناتك الأولية الخاصة لهذه المرحلة، راجع القسم الخاص بالإضافات وتخصيص مديري التمريرات.
## مرحلة Layout
تتعلق المرحلة التالية بتخطيط أو اتصالية الـ Backend الذي ستُرسَل إليه الدائرة. بشكل عام، الدوائر الكمومية كيانات مجردة تكون فيها الكيوبتات "افتراضية" أو "منطقية" كتمثيل للكيوبتات الفعلية المستخدمة في الحسابات. لتنفيذ سلسلة من البوابات، يلزم وجود تعيين واحد لواحد من الكيوبتات "الافتراضية" إلى الكيوبتات "المادية" في جهاز كمومي فعلي. يُخزَّن هذا التعيين ككائن `Layout` وهو جزء من القيود المحددة ضمن [بنية مجموعة تعليمات الـ Backend (ISA)](./transpile#instruction-set-architecture).

![تُوضّح هذه الصورة تعيين الكيوبتات من تمثيل السلك إلى مخطط يمثل كيفية اتصال الكيوبتات في وحدة المعالجة الكمومية.](../docs/images/guides/transpiler-stages/layout-mapping.avif "تعيين الكيوبتات")

يُعدّ اختيار التعيين بالغ الأهمية لتقليل عدد عمليات SWAP المطلوبة لتعيين دائرة الإدخال على طبولوجيا الجهاز وضمان استخدام الكيوبتات الأكثر دقة في المعايرة. نظرًا لأهمية هذه المرحلة، يجرّب مديرو التمريرات المُعدّون مسبقًا عدة طرق مختلفة للعثور على أفضل تخطيط. عادةً ما يتضمن ذلك خطوتين: أولًا، محاولة العثور على تخطيط "مثالي" (تخطيط لا يتطلب أي عمليات SWAP)، ثم تمريرة بالاستدلال تحاول العثور على أفضل تخطيط للاستخدام إذا لم يُعثر على تخطيط مثالي. هناك `Passes` يُستخدم عادةً للخطوة الأولى:

- `TrivialLayout`: يُعيِّن كل كيوبت افتراضي ببساطة إلى الكيوبت المادي بالرقم نفسه على الجهاز (أي [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]). هذا سلوك تاريخي يُستخدم فقط في `optimzation_level=1` لمحاولة إيجاد تخطيط مثالي. إذا فشل، تُجرَّب `VF2Layout` بعده.
- `VF2Layout`: هذه `AnalysisPass` تختار تخطيطًا مثاليًا من خلال معالجة هذه المرحلة كمسألة تشاكل الرسم الجزئي، يُحلّها خوارزمية VF2++. إذا وُجد أكثر من تخطيط واحد، تُشغَّل استدلالية تسجيل لاختيار التعيين ذي أقل متوسط خطأ.

ثم في مرحلة الاستدلال، تُستخدم تمريرتان افتراضيًا:

- `DenseLayout`: تجد الرسم الفرعي للجهاز ذا الاتصالية الأعلى والذي يحتوي على نفس عدد الكيوبتات في الدائرة (تُستخدم لمستوى التحسين 1 إذا كانت هناك عمليات تدفق تحكم مثل IfElseOp موجودة في الدائرة).
- `SabreLayout`: تختار هذه التمريرة تخطيطًا بالبدء من تخطيط عشوائي أولي وتشغيل خوارزمية `SabreSwap` بشكل متكرر. تُستخدم هذه التمريرة فقط في مستويات التحسين 1 و2 و3 إذا لم يُعثر على تخطيط مثالي عبر تمريرة `VF2Layout`. لمزيد من التفاصيل حول هذه الخوارزمية، راجع الورقة البحثية [arXiv:1809.02573.](https://arxiv.org/abs/1809.02573)
## مرحلة Routing
لتطبيق بوابة ثنائية الكيوبت بين كيوبتات غير متصلة مباشرةً على جهاز كمومي، يجب إدراج بوابة SWAP واحدة أو أكثر في الدائرة لنقل حالات الكيوبت حتى تصبح متجاورة على خريطة البوابات في الجهاز. كل بوابة SWAP تمثل عملية مكلفة وضوضائية. وبالتالي، يُعدّ إيجاد الحد الأدنى من بوابات SWAP اللازمة لتعيين دائرة على جهاز معين خطوة مهمة في عملية التحويل. لأسباب كفاءة، تُحسَب هذه المرحلة عادةً جنبًا إلى جنب مع مرحلة Layout افتراضيًا، لكنهما متمايزتان منطقيًا. تختار مرحلة *Layout* الكيوبتات المادية المراد استخدامها، بينما تُدرج مرحلة *Routing* القدر المناسب من بوابات SWAP لتنفيذ الدوائر باستخدام التخطيط المختار.

ومع ذلك، يصعب إيجاد أمثل تعيين SWAP. في الواقع، هي مسألة NP-Hard، وبالتالي باهظة التكلفة الحسابية لجميع الأجهزة الكمومية ودوائر الإدخال إلا الأصغر منها. للتغلب على ذلك، يستخدم Qiskit خوارزمية استدلالية عشوائية تسمى `SabreSwap` لحساب تعيين SWAP جيد لكن ليس بالضرورة الأمثل. يعني استخدام الأسلوب العشوائي أن الدوائر الناتجة غير مضمونة لتكون متطابقة عبر تشغيلات متكررة. في الواقع، يُنتج تشغيل الدائرة نفسها مرارًا توزيعًا من أعماق الدوائر وأعداد البوابات في المخرجات. لهذا السبب، يختار كثير من المستخدمين تشغيل دالة التوجيه (أو `StagedPassManager` بأكملها) مرات عديدة واختيار الدوائر ذات أقل عمق من توزيع المخرجات.

على سبيل المثال، لنأخذ دائرة GHZ من 15 كيوبت تُنفَّذ 100 مرة، باستخدام `initial_layout` "سيء" (غير متصل).

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.providers.fake_provider import GenericBackendV2

backend = GenericBackendV2(15)


ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/62479697-cef1-4a89-ac2a-20051dd294f4-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ab89c4ea-06c4-4320-b493-feb691b3570d-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/f6a3a92a-8656-4518-ba2c-c3b0b038f507-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](/docs/guides/transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/d4d1f65a-3336-4d70-9189-65ba010f2366-1.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

كما ترى، تضطر هذه الدائرة إلى تنفيذ بوابة ثنائية الكيوبت بين الكيوبتين 0 و14، اللذين يقعان في أماكن بعيدة جدًا على رسم الاتصالية. وبالتالي، يتطلب تشغيل هذه الدائرة إدراج بوابات SWAP لتنفيذ جميع البوابات ثنائية الكيوبت باستخدام تمريرة `SabreSwap`.

لاحظ أيضًا أن خوارزمية `SabreSwap` تختلف عن أسلوب `SabreLayout` الأشمل في المرحلة السابقة. افتراضيًا، يُشغِّل `SabreLayout` كلًا من التخطيط والتوجيه، ويُعيد الدائرة المحوَّلة. يُنجز ذلك لأسباب تقنية محددة مبيّنة في [صفحة مرجع API](../api/qiskit/qiskit.transpiler.passes.SabreLayout) الخاصة بالتمريرة.
## مرحلة Translation
عند كتابة دائرة كمومية، أنت حر في استخدام أي بوابة كمومية (عملية أحادية) تريدها، إلى جانب مجموعة من العمليات غير البوابية مثل تعليمات قياس الكيوبت أو إعادة ضبطه. ومع ذلك، تدعم معظم الأجهزة الكمومية أصلًا عددًا محدودًا من عمليات البوابات الكمومية وغيرها. هذه البوابات الأصيلة جزء من تعريف [ISA](./transpile#instruction-set-architecture) الهدف، وتُترجم هذه المرحلة من `PassManagers` المُعدَّة مسبقًا البوابات المحددة في الدائرة (أو *تُفكّكها*) إلى البوابات الأساسية الأصيلة للـ Backend المحدد. هذه خطوة مهمة تتيح تنفيذ الدائرة بواسطة الـ Backend، لكنها عادةً تؤدي إلى زيادة العمق وعدد البوابات.

ثمة حالتان خاصتان مهمتان بشكل استثنائي توضحان ما تفعله هذه المرحلة.

1. إذا لم تكن بوابة SWAP بوابة أصيلة للـ Backend المستهدف، فهذا يتطلب ثلاث بوابات CNOT:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4552b367-75e9-4faa-a59d-8317e25ac145-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2700405a-f559-45d3-99a9-5b4447621743-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

بما أنها حاصل ثلاث بوابات CNOT، فإن SWAP تُعدّ عملية مكلفة لأداء على الأجهزة الكمومية الضوضائية. ومع ذلك، فإن مثل هذه العمليات ضرورية عادةً لتضمين دائرة ضمن اتصالية البوابات المحدودة لكثير من الأجهزة. وبالتالي، يُعدّ تقليل عدد بوابات SWAP في الدائرة هدفًا رئيسيًا في عملية التحويل.

2. بوابة Toffoli، أو بوابة النفي التحكمي المضاعف (`ccx`)، هي بوابة ثلاثية الكيوبت. نظرًا لأن مجموعة البوابات الأساسية لدينا تشمل بوابات الكيوبت الواحد والكيوبتين فقط، يجب تفكيك هذه العملية. ومع ذلك، فإنها مكلفة للغاية:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = GenericBackendV2(5)

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ae2d9390-5c26-46f3-9418-a684ba8a406a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

لكل بوابة Toffoli في دائرة كمومية، قد ينفذ العتاد ما يصل إلى ست بوابات CNOT وعدد من بوابات الكيوبت الواحد. يُبرهن هذا المثال على أن أي خوارزمية تستخدم بوابات Toffoli متعددة ستنتهي كدائرة ذات عمق كبير وستتأثر بالضوضاء تأثيرًا ملحوظًا.
## مرحلة Optimization
تتمحور هذه المرحلة حول تفكيك الدوائر الكمومية إلى مجموعة البوابات الأساسية للجهاز المستهدف، ويجب أن تواجه الزيادة في العمق الناتجة عن مرحلتي التخطيط والتوجيه. لحسن الحظ، هناك روتينات كثيرة لتحسين الدوائر إما بدمج البوابات أو حذفها. في بعض الحالات، تكون هذه الأساليب فعّالة لدرجة أن الدوائر الناتجة أقل عمقًا من الإدخالات، حتى بعد التخطيط والتوجيه على طبولوجيا العتاد. في حالات أخرى، لا يمكن فعل الكثير، وقد تكون الحسابات صعبة التنفيذ على الأجهزة الضوضائية. هنا تبدأ مستويات التحسين المختلفة في التباين.

- في `optimization_level=1`، تُعدّ هذه المرحلة [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) و[`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation)، اللتان تدمجان سلاسل بوابات الكيوبت الواحد وتُلغيان أي بوابات CNOT متتالية.
- في `optimization_level=2`، تستخدم هذه المرحلة تمريرة [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) بدلًا من `CXCancellation`، التي تحذف البوابات الزائدة باستغلال علاقات التبادلية.
- في `optimization_level=3`، تُعدّ هذه المرحلة التمريرات التالية:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

علاوةً على ذلك، تُنفِّذ هذه المرحلة أيضًا بعض الفحوصات النهائية للتأكد من أن جميع التعليمات في الدائرة مؤلَّفة من البوابات الأساسية المتاحة على الـ Backend المستهدف.

يوضح المثال التالي باستخدام حالة GHZ تأثيرات إعدادات مستوى التحسين المختلفة على عمق الدائرة وعدد البوابات.

> **Note:** تتباين مخرجات التحويل بسبب مُعيِّن SWAP العشوائي. لذا، من المرجح أن تتغير الأرقام أدناه في كل مرة تُشغِّل فيها الكود.

![حالة GHZ من 15 كيوبت](../docs/images/guides/transpiler-stages/transpiler-11.avif "حالة GHZ من 15 كيوبت قبل التحويل")

يبني الكود التالي حالة GHZ من 15 كيوبت ويقارن `optimization_levels` للتحويل من حيث عمق الدائرة الناتجة وأعداد البوابات وأعداد البوابات متعددة الكيوبتات.